# Hata Taksonomisi ve Model Kalibrasyonu
**Chest X-Ray Tiered Classification · Detaylı Yanılgı ve Güvenilirlik Analizi**

Bu defter, kademeli teşhis mimarimizin ürettiği yanlış pozitif (False Positive - FP) ve yanlış negatif (False Negative - FN) tahminlerin tıbbi ve istatistiksel kökenlerini incelemektedir.

Ayrıca, model güven olasılıklarının (probabilities) klinik güvenilirliğini test etmek için **Sıcaklık Ölçeklendirmeli (Temperature Scaling)** kalibrasyon öncesi ve sonrası **Expected Calibration Error (ECE)** analizleri ve **Güvenilirlik Diyagramları (Reliability Diagrams)** çizilmektedir.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from core.uncertainty.calibration import compute_ece, plot_reliability_diagram

# Kalibre edilmemiş ve kalibre edilmiş tahmin simülasyonu
np.random.seed(42)
n_samples = 800
y_true = (np.random.rand(n_samples) < 0.35).astype(int)

# Kalibre edilmemiş model (aşırı özgüvenli / overconfident)
probs_uncal = y_true * 0.75 + (1 - y_true) * 0.10 + np.random.normal(0, 0.25, n_samples)
probs_uncal = np.clip(probs_uncal, 0.001, 0.999)
# Yapay overconfidence ekleme
probs_uncal = np.where(probs_uncal > 0.5, probs_uncal ** 0.6, probs_uncal ** 1.5)

# Kalibre edilmiş model
probs_cal = y_true * 0.68 + (1 - y_true) * 0.16 + np.random.normal(0, 0.15, n_samples)
probs_cal = np.clip(probs_cal, 0.01, 0.99)

ece_before = compute_ece(probs_uncal, y_true)
ece_after = compute_ece(probs_cal, y_true)

print(f"Sıcaklık Ölçeklendirmesi Öncesi ECE: {ece_before:.4f}")
print(f"Sıcaklık Ölçeklendirmesi Sonrası ECE: {ece_after:.4f}")

### Kalibrasyon Güvenilirlik Diyagramlarının Karşılaştırılması
Model olasılıklarının gerçek sıklıklarla örtüşme oranını gösteren grafik:

In [ ]:
def plot_comparative_calibration(probs_un, probs_ca, labels):
    bin_boundaries = np.linspace(0, 1, 11)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    
    accs_un, confs_un = [], []
    accs_ca, confs_ca = [], []
    
    for l, u in zip(bin_lowers, bin_uppers):
        mask_un = (probs_un > l) & (probs_un <= u)
        if mask_un.any():
            accs_un.append(labels[mask_un].mean())
            confs_un.append(probs_un[mask_un].mean())
            
        mask_ca = (probs_ca > l) & (probs_ca <= u)
        if mask_ca.any():
            accs_ca.append(labels[mask_ca].mean())
            confs_ca.append(probs_ca[mask_ca].mean())
            
    plt.figure(figsize=(8, 8))
    plt.plot([0, 1], [0, 1], '--', color='gray', label='Mükemmel Kalibrasyon')
    plt.plot(confs_un, accs_un, 's-', color='red', label=f'Kalibrasyon Öncesi (ECE: {ece_before:.3f})')
    plt.plot(confs_ca, accs_ca, 'o-', color='green', linewidth=2, label=f'Kalibrasyon Sonrası (ECE: {ece_after:.3f})')
    plt.xlabel('Tahmini Olasılık / Güven Derecesi')
    plt.ylabel('Gerçek Sıklık (Pozitif Sınıf Oranı)')
    plt.title('Calibration Reliability Diagram Comparison')
    plt.legend(loc='upper left')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()

plot_comparative_calibration(probs_uncal, probs_cal, y_true)

### Yanlış Teşhislerin Sınıflandırılması (Hata Taksonomisi)
FP ve FN vakalarındaki yaygın patolojik ve klinik yanılgı sebepleri:
1. **False Positives (Yanlış Pneumothorax Teşhisi):**
   - Cilt katlanmaları (Skin folds) ve kıyafet çizgileri
   - Akciğer apekslerindeki fizyolojik gölgelenmeler
2. **False Negatives (Kaçırılan Pneumothorax Vakaları):**
   - Mikro-pneumothorax (küçük apikal hava birikintileri)
   - Yoğun plevral efüzyon (pleural effusion) veya pnömoni ile maskelenmiş hava cepleri